In [9]:
# First we initialize the model we want to use.
from langchain_openai import ChatOpenAI
from langchain_community.tools import TavilySearchResults
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from typing import List
from langgraph.graph import MessagesState

from langchain_core.pydantic_v1 import BaseModel, Field

class GlobalContext(BaseModel):
    """Respond to the user with this"""
    global_context: List[str] = Field(description="List of context elements (results from the web search)")

tavily_tool = TavilySearchResults(
    max_results=5,
    include_answer=True,
)

tools = [tavily_tool, GlobalContext]

model_with_response_tool = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(
    tools, tool_choice="any", parallel_tool_calls=False
)


# Define the AgentState
class AgentState(MessagesState):
    final_response: GlobalContext


# Define the function that calls the model
def call_model(state: AgentState):
    response = model_with_response_tool.invoke(state["messages"])
    return {"messages": [response]}


# Define the function that responds to the user
def respond(state: AgentState):
    response = GlobalContext(**state['messages'][-1].tool_calls[0]['args'])
    # We return the final answer
    return {"final_response": response}


# Define the function that determines whether to continue or not
def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    # If there is only one tool call and it is the response tool call we respond to the user
    if len(last_message.tool_calls) == 1 and last_message.tool_calls[0]['name'] == "GlobalContext":
        return "respond"
    # Otherwise we will use the tool node again
    else:
        return "continue"

In [10]:
# Define a new graph
workflow = StateGraph(AgentState)

# Define the nodes
workflow.add_node("agent", call_model)
workflow.add_node("respond", respond)
workflow.add_node("tools", ToolNode(tools))

# Set the entrypoint as `agent`
workflow.set_entry_point("agent")

# Add conditional edges
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "tools",
        "respond": "respond",
    },
)

workflow.add_edge("tools", "agent")
workflow.add_edge("respond", END)

# Compile the graph
graph = workflow.compile()


In [13]:
from langchain_core.prompts import ChatPromptTemplate

# Function to print the stream
def print_stream(stream):
    for s in stream:
        if 'messages' in s:
            message = s["messages"][-1]
            if isinstance(message, tuple):
                print(message)
            else:
                message.pretty_print()
        elif 'final_response' in s:
            print("Final Response:", s['final_response'])

# Read raw_job_posting.txt
with open("raw_job_posting.txt", "r") as f:
    raw_job_posting = f.read()

from langchain import hub
hub_prompt = hub.pull("scorecard-enrichment-react-basic")

# Convert the hub prompt to a ChatPromptTemplate
chat_prompt = ChatPromptTemplate.from_messages(hub_prompt.messages)

# Format the messages
formatted_messages = chat_prompt.format_messages(raw_job_posting=raw_job_posting)

# Create the input dictionary
inputs = {
    "messages": formatted_messages
}

# Stream the graph with the correct input format
print_stream(graph.stream(inputs, stream_mode="values"))

================================ Human Message =================================

Begin!

<raw_job_posting> 
Ynstant
Operations Manager CDI Paris Salaire :52K à 72K € Début :19 août 2024 Télétravail occasionnel Expérience :> 2 ans Compétences & expertises Compétences en communication Outils d'automatisation Sensibilité culturelle Pandas Sql
entreprise : Ynstant

Ynstant est une startup qui révolutionne la mobilité du quotidien pour la rendre plus durable, grâce à une application de covoiturage instantané. Concrètement, Ynstant permet aux conducteurs de covoiturer en deux clics, sans détour et au dernier moment avant de partir, préservant ainsi leur flexibilité.

Descriptif du poste
Mission : Le Ops Manager sera responsable de la gestion des opérations quotidiennes des Certificats d’Economie d’Energie (CEE) en s’appuyant sur de la data. Les missions sont :
-Monitoring des dossiers CEE et optimisation de chaque étape du parcours
-Traitement des dossiers présentant des anomalies
-Conformi

In [2]:
# Read raw_job_posting.txt
with open("raw_job_posting.txt", "r") as f:
    raw_job_posting = f.read()

In [3]:

from scorecard.sub_graphs.enrichment.node_generate_queries import node_generate_queries
from scorecard.sub_graphs.enrichment.state import EnrichmentGraphState

state = EnrichmentGraphState(raw_job_posting=raw_job_posting, queries=[], results=[])

res = node_generate_queries(state)
print(res)

/Users/aberman/Documents/Workshop/repio-intelligence/.venv/lib/python3.12/site-packages/langsmith/client.py:5301: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))
/Users/aberman/Documents/Workshop/repio-intelligence/src/scorecard/sub_graphs/enrichment/node_generate_queries.py:19: LangChainBetaWarning: The function `init_chat_model` is in beta. It is actively being worked on, so the API may change.
  model = init_chat_model(


{'queries': ['Key performance indicators for Operations Manager in energy efficiency sector', "Certificats d'Economie d'Energie (CEE) regulations and compliance requirements", 'Best practices for data-driven operations management in startups', 'Automation tools and scripts for anomaly detection in energy efficiency certificates', 'Cultural fit assessment techniques for startup operations roles']}
